In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path


class MediumTermMemory:
    """
    Append-only table-scoped buffer that accumulates hand records and trend
    observations across an entire game (one session at a single table).
    Order of calls:
        new_game() called by the game engine when a new table session begins
        ingest_hand() called after each hand with the raw record from ShortTermMemory.purge_memory(); appends raw log and bumps stats
        log_trend() called by the agent after reflecting on a hand to record a human-readable trend observation (e.g. "P2 has 3-bet preflop
                    in 4 of 6 hands — likely a wide 3-bet range")
        read_digest() called by the agent at the start of each turn; returns a structured summary (stats + recent trends) rather than the
                    full raw log, keeping context window usage bounded
        read_raw() returns the full unabridged game log (for end-of-game reflection)
        close_game() seals the buffer with final chip counts and a game-level critique
        purge_memory() wipes the buffer; returns the full game record for long-term ingestion

    File layout under buffer_dir/:
        game_<id>/
            raw_log.txt every hand record, appended verbatim
            trends.txt agent trend observations, one per hand
            stats.json running numeric stats (hands played, vpip counts, etc.)
            metadata.json game-level metadata (start time, players, game_id)
    """

    def __init__(self, buffer_dir: str = "./memory/medium_term"):
        self.buffer_dir = Path(buffer_dir)
        self.buffer_dir.mkdir(parents=True, exist_ok=True)
        self._game_dir: Path | None = None
        self.game_id: str = ""
        self.hands_played: int = 0


    def new_game(self, game_id: str, players: list[str]):
        """
        Begin a new game session.
        Args:
        game_id:  unique identifier for this table session (e.g. "game_001")
        players:  list of player labels at the table, e.g. ["Hero", "P2", "P3"]
        """
        self.game_id = game_id
        self.hands_played = 0
        self._game_dir = self.buffer_dir / game_id
        self._game_dir.mkdir(parents=True, exist_ok=True)

        metadata = {
            "game_id": game_id,
            "started_at": datetime.now().isoformat(),
            "players": players,
        }
        self._write_json("metadata.json", metadata)

        #for each player:
        stats = {
            "hands_played": 0,
            "players": {p: self._empty_player_stats() for p in players},
        }
        self._write_json("stats.json", stats)

        # log games
        self._append_to("raw_log.txt",
            f"=== GAME {game_id} STARTED | {metadata['started_at']} ===\n"
            f"Players: {', '.join(players)}\n\n"
        )
        self._append_to("trends.txt",
            f"=== TREND LOG | GAME {game_id} ===\n\n"
        )

    def ingest_hand(self, action_history: list[str], reasoning: list[list], chip_changes: list[int])->None:
        """
        Alternative method to ingest hand to deprecate shortterm memory. Instead,
        read from the json and the reasoning list.
        """
        self.hands_played += 1
        #action history is a list of string
        for item in action_history:
            self._append_to("raw_log.txt", f'{item}\n')

        self._append_to("raw_log.txt", '\nReasonings:\n')
        streets = ["Preflop", "Flop", "Turn", "River"]
        for reasoning_on_street, streetname in zip(reasoning, streets):
            self._append_to("raw_log.txt", f'{streetname}: {reasoning_on_street}')

        stats = self._read_json("stats.json")
        stats["hands_played"] = self.hands_played
        player_names = list(stats.get("players", {}).keys())
        for name, change in zip(player_names, chip_changes):
            stats["players"][name]["net_chips"] = stats["players"][name].get("net_chips", 0) + int(change)

        self._write_json("stats.json", stats)



    def log_trend(self, observation: str, hand_number: int | None = None) -> None:
        """
        Record a free-prose trend observation after hand-level reflection.

        Args:
        observation: the agent's natural-language finding, e.g.:
            "P2 has continuation-bet the flop in 3/4 opportunities — treat flop
             c-bets from P2 as weak until proven otherwise."
        hand_number: if provided, prefixes the entry for traceability.
        """
        hand_ref = f"[After hand #{hand_number}]" if hand_number is not None else f"[After hand #{self.hands_played}]"
        entry = f"{hand_ref}\n{observation.strip()}\n\n"
        self._append_to("trends.txt", entry)


    def read_digest(self, recent_trends: int = 5):
        """
        Return a bounded summary suitable for injection into the agent's context
        at the start of each turn.

        Includes:
          - Running numeric stats (frequencies derived from stats.json)
          - The N most recent trend observations
        unlike read_raw(), this is a summary

        Args:
        recent_trends: how many of the latest trend entries to include (default 5).
        """
        if not self._game_dir:
            return "[Medium term memory is empty — no active game]"

        stats = self._read_json("stats.json")
        n = max(stats.get("hands_played", 0), 1) #div/0 error sentinel

        lines: list[str] = [f"=== LIVE TABLE DIGEST | Game {self.game_id} | {n} hands played ===\n"]

        for player, bucket in stats.get("players", {}).items():
            vpip_pct = 100 * bucket.get("vpip", 0)/n
            pfr_pct = 100 * bucket.get("preflop_raise", 0)/n
            pf3b_pct = 100 * bucket.get("preflop_3bet", 0)/n
            cbet_pct = 100 * bucket.get("continuation_bet", 0)/n
            wtsd_pct = 100 * bucket.get("went_to_showdown", 0)/n
            net = bucket.get("net_chips", 0)
            lines.append(
                f"  {player}: VPIP={vpip_pct:.0f}% | PFR={pfr_pct:.0f}% | "
                f"3B={pf3b_pct:.0f}% | CBet={cbet_pct:.0f}% | "
                f"WTSD={wtsd_pct:.0f}% | Net={net:+d} chips"
            )
        lines.append("")

        trends_text = self._read_text("trends.txt")
        #sptit on \n\n means we split on blank single lines.
        entries = [e.strip() for e in trends_text.split("\n\n") if e.strip() and not e.strip().startswith("=== TREND LOG")]
        if entries:
            lines.append(f"--- Recent Observations (last {recent_trends}) ---")
            for entry in entries[-recent_trends:]:
                lines.append(entry)
        else: lines.append("--- No trend observations recorded yet ---")

        return "\n".join(lines)

    def read_raw(self):
        """Return the full unabridged game log. Used for end-of-game reflection."""
        if not self._game_dir:
            return "[Medium term memory is empty — no active game]"
        return self._read_text("raw_log.txt")


    def close_game(self, final_stacks: dict, game_critique: str = ""):
        """
        Seals the buffer with final results and an optional game-level critique.
        Args:
        final_stacks: e.g. {"Hero": 620, "P2": 430, "P3": 450}
        game_critique: agent's high-level reflection on the session
        """
        stacks_str = " | ".join(f"{p}: {c}" for p, c in final_stacks.items())
        footer = (
            f"\n=== GAME {self.game_id} CLOSED | {datetime.now().isoformat()} ===\n"
            f"Final stacks: {stacks_str}\n"
            f"Hands played: {self.hands_played}\n"
            f"Game critique: {game_critique.strip() or 'pending'}\n"
            f"{'=' * 60}\n"
        )
        self._append_to("raw_log.txt", footer)
        self._append_to("trends.txt", footer)

    def purge_memory(self):
        """
        Return the full game record (raw log + trends concatenated) and wipe
        the game directory from disk.
        The returned string is what gets handed to the long-term reflection
        pipeline for player-profile updates.
        """
        if not self._game_dir:
            return "[Nothing to purge — no active game]"

        raw = self._read_text("raw_log.txt")
        trends = self._read_text("trends.txt")
        combined = f"{raw}\n\n{'=' * 60}\n\n{trends}"

        # Wipe game directory.
        for child in self._game_dir.iterdir():
            child.unlink()
        self._game_dir.rmdir()

        self._game_dir = None
        self.game_id = ""
        self.hands_played = 0

        return combined



  #internal class methods below

    @staticmethod
    def _empty_player_stats():
        return {
            "vpip": 0,
            "preflop_raise": 0,
            "preflop_3bet": 0,
            "continuation_bet": 0,
            "went_to_showdown": 0,
            "won_hand": 0,
            "net_chips": 0,
        }

    def _path(self, filename: str) -> Path:
        assert self._game_dir is not None, "Call new_game() before accessing memory."
        return self._game_dir / filename

    def _append_to(self, filename: str, text: str) -> None:
        with open(self._path(filename), "a", encoding="utf-8") as f:
            f.write(text)

    def _read_text(self, filename: str) -> str:
        p = self._path(filename)
        if not p.exists():
            return ""
        return p.read_text(encoding="utf-8")

    def _read_json(self, filename: str) -> dict:
        p = self._path(filename)
        if not p.exists():
            return {}
        try:
            return json.loads(p.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            return {}

    def _write_json(self, filename: str, data: dict) -> None:
        with open(self._path(filename), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

In [ ]:
!pip install pokerkit
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.7/113.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.6/699.6 kB 13.1 MB/s eta 0:00:00


In [ ]:
import random
from pokerkit import *
from dotenv import load_dotenv
import json
from anthropic import Anthropic, beta_tool
from google.colab import userdata

In [ ]:
load_dotenv()

client = Anthropic(api_key=userdata.get("anthropicapi"))

In [ ]:
system_prompt = """You are a poker agent playing 3-handed Texas Hold'em No-Limit.
Blinds are 1000/2000. You are Player 1 (Hero). Player 2 is P2, and Player 3 is P3.
Think about pot odds, position, and hand strength before deciding. Make sure to carefully read and use the information in the provided json
Use the take_action tool to submit your decision."""

@beta_tool
def take_action(action: str, reasoning: str, amount: float = 0) -> str:
    """Submit your poker action for this turn and give a maximum 3 sentence explanation for why.

    Args:
        action: One of fold, check, call, raise
        amount: Raise amount in chips. Required if action is raise.
        reasoning: Reasoning for why you took a poker action. Maximum 3 sentences. Rate how aggressive each opponent's actions are on a 1-10 scale, but only if you made it to the flop or later.
    """
    return json.dumps({"status": "accepted", "action": action, "amount": amount, 'reasoning': reasoning})

def run_turn(state_json):
    messages = [{
        "role": "user",
        "content": f"""It's your turn to act. Here is the current game state:

    ```json
    {state_json}
    ```

    Decide your action."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[take_action],
        system=system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        tokens = f"Input: {message.usage.input_tokens}, Output: {message.usage.output_tokens}"

        for block in message.content:
            if block.type == "tool_use" and block.name == "take_action":
                agent_action = block.input

    return agent_action, tokens

def street_converter(index):
    if index == 0:
        return 'preflop'
    elif index == 1:
        return 'flop'
    elif index == 2:
        return 'turn'
    elif index == 3:
        return 'river'

def get_visible_operations(state):
    visible = []
    showdown = sum(1 for p in state.statuses if p) > 1
    for op in state.operations:
        if isinstance(op, ChipsPushing):
            winners = {i for i, amt in enumerate(op.amounts) if amt > 0}

    for op in state.operations:
        if isinstance(op, HoleDealing):
            continue
        elif isinstance(op, CardBurning):
            continue
        elif isinstance(op, HoleCardsShowingOrMucking):
            if showdown or op.player_index in winners:
                visible.append(op)
        else:
            visible.append(op)
    return visible

judge_system_prompt = """You are an evaluating agent for players playing Texas Hold-em. Your job is to evaluate and provide useful information about the other non-agent players to aid Player 1 in defeating them. Do not evaluate Player 1 (Hero), focus on the remaining players. Remember that index 0 is the agent, index 1 is Player 2 and so on. Use the generate_observation tool to submit your decision."""

@beta_tool
def generate_observation(evaluation: str) -> str:
    """Evaluate the action history of the game in the previous poker hand.

    Args:
        evaluation: The evaluation of how the other players played. Be specific as to any mistakes or good moves each non-Hero player made.
    """
    return json.dumps({"status": "accepted", "evaluation": evaluation})


def run_observation_generator(action_history):
    eval_json = {'action_history': action_history}
    messages = [{
        "role": "user",
        "content": f"""Here is the action history of the game.

    ```json
    {eval_json}
    ```

    Generate your observations."""}]

    runner = client.beta.messages.tool_runner(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        tools=[generate_observation],
        system=judge_system_prompt,
        messages=messages
    )

    agent_action = None

    for message in runner:
        for block in message.content:
            if block.type == "tool_use" and block.name == "generate_observation":
                agent_action = block.input["evaluation"]

    return agent_action


In [ ]:
# Full game
med = MediumTermMemory()
med.new_game(game_id = '0', players = ['Hero', 'P2', 'P3', 'P4'])
# Initial Setup

def pocket_pair_strategy(state, player_index):
    hole = state.hole_cards[player_index]
    is_pair = len(hole) >= 2 and hole[0].rank == hole[1].rank
    strong_pair_ranks = {"A", "K", "Q", "J", "T", "9", "8", "7"}
    is_strong_pair = is_pair and hole[0].rank in strong_pair_ranks
    if is_strong_pair and state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif is_pair and state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()
    else:
        state.check_or_call()

def punter_strategy(state, player_index):
    if not punter_is_punting_this_hand:
        if state.can_fold():
            state.fold()
        else:
            state.check_or_call()
        return
    if state.can_complete_bet_or_raise_to():
        min_raise = state.min_completion_betting_or_raising_to_amount
        max_raise = state.max_completion_betting_or_raising_to_amount
        target_raise = min(min_raise * 2, max_raise)
        state.complete_bet_or_raise_to(target_raise)
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

def tight_passive_strategy(state, player_index):
    r = random.random()
    if r < 0.9 and state.can_fold():
        state.fold()
    elif state.can_check_or_call():
        state.check_or_call()
    elif state.can_fold():
        state.fold()

opponent_strategies = {
    1: pocket_pair_strategy,
    2: punter_strategy,
    3: tight_passive_strategy,
}

player_stacks = [10_000, 10_000, 10_000, 10_000] # Starting stacks for 3 players [cite: 85]
hand_count = 0

# The Global Game Loop: Runs until only one player has chips
while len([s for s in player_stacks if s > 0]) > 1:
    reasonings = []
    active_indices = [i for i, stack in enumerate(player_stacks) if stack > 0]

    # If only one player is left, the game is over
    if len(active_indices) < 2:
        break

# Filter stacks for the engine
    current_stacks = tuple(player_stacks[i] for i in active_indices)
    hand_count += 1
    print(f"\n--- STARTING HAND {hand_count} ---")

    state = NoLimitTexasHoldem.create_state(
        (
            Automation.ANTE_POSTING,
            Automation.BET_COLLECTION,
            Automation.BLIND_OR_STRADDLE_POSTING,
            Automation.CARD_BURNING,
            Automation.HOLE_DEALING,
            Automation.BOARD_DEALING,
            Automation.HOLE_CARDS_SHOWING_OR_MUCKING,
            Automation.HAND_KILLING,
            Automation.CHIPS_PUSHING,
            Automation.CHIPS_PULLING,
        ),
        True, 0, (1000, 2000), 2000, tuple(current_stacks), len(active_indices)
    )
    #big blind ante, ante, blinds, min bet, stacks, players
    punter_is_punting_this_hand = random.random() < 0.1

    # Internal Decision Loop (Per-Turn) [cite: 19, 49]
    while state.status:
        if state.can_deal_hole():
            state.deal_hole()
        elif state.can_deal_board():
            state.deal_board()

        elif state.actor_index is not None:
            res = get_visible_operations(state)
            obs = {
                "your_index": 0,
                "pot": state.total_pot_amount,
                "board": state.board_cards,
                "hole": state.hole_cards[state.actor_index],
                "stacks": state.stacks,
                "bets": state.bets,
                "street": street_converter(state.street_index),
                "can_fold?": state.can_fold(),
                "can_check_or_call?": state.can_check_or_call(),
                "can_raise?": state.can_complete_bet_or_raise_to(),
                "min_raise": state.min_completion_betting_or_raising_to_amount,
                "max_raise": state.max_completion_betting_or_raising_to_amount,
                "how_much_to_call": state.checking_or_calling_amount,          # how much a call costs
                "action_history": res
            }
            if state.actor_index == 0: # Agent
                # print(obs)
                action = run_turn(obs)
                if action[0]['action'] in ['check', 'call']:
                    state.check_or_call()
                elif action[0]['action'] == 'raise':
                    state.complete_bet_or_raise_to(action[0]['amount'])
                elif action[0]['action'] == 'fold':
                    state.fold()
                try:
                    reasonings.append([action[0]['action'], action[0]['amount'], action[0]['reasoning']])
                except:
                    reasonings.append([action[0]['action'], 0, action[0]['reasoning']])
            else:
                global_player_index = active_indices[state.actor_index]
                opponent_strategies[global_player_index](state, state.actor_index)
        else:
            break

    # Update player stacks for the next hand based on payoffs [cite: 91, 92]
    # payoffs return the net change (e.g., [-2000, 4000, -2000])
    full_chip_changes = [0] * len(player_stacks)
    for i, global_idx in enumerate(active_indices):
        full_chip_changes[global_idx] = int(state.payoffs[i])
    med.ingest_hand(action_history=list(map(lambda x: str(x), get_visible_operations(state))),
                    reasoning=reasonings, chip_changes=full_chip_changes)
    eval = run_observation_generator(get_visible_operations(state))
    med.log_trend(eval)
    # print(reasonings)
    for i, global_idx in enumerate(active_indices):
        player_stacks[global_idx] += int(state.payoffs[i])

    print(f"Hand {hand_count} Results: {state.payoffs}")
    print(f"New Stacks: {player_stacks}")

print(f"\nGAME OVER after {hand_count} hands!")
print(f"Final Winner Stacks: {player_stacks}")


--- STARTING HAND 1 ---
Hand 1 Results: [-2000, 2000, 0, 0]
New Stacks: [8000, 12000, 10000, 10000]

--- STARTING HAND 2 ---
Hand 2 Results: [-2000, 2000, 0, 0]
New Stacks: [6000, 14000, 10000, 10000]

--- STARTING HAND 3 ---
Hand 3 Results: [-1000, -2000, 0, 3000]
New Stacks: [5000, 12000, 10000, 13000]

--- STARTING HAND 4 ---
Hand 4 Results: [-2000, -2000, 0, 4000]
New Stacks: [3000, 10000, 10000, 17000]

--- STARTING HAND 5 ---
Hand 5 Results: [-2000, 2000, 0, 0]
New Stacks: [1000, 12000, 10000, 17000]

--- STARTING HAND 6 ---
Hand 6 Results: [-1000, 1000, 0, 0]
New Stacks: [0, 13000, 10000, 17000]

--- STARTING HAND 7 ---
Hand 7 Results: [-1000, -2000, 3000]
New Stacks: [0, 12000, 8000, 20000]

--- STARTING HAND 8 ---
Hand 8 Results: [-1000, 3000, -2000]
New Stacks: [0, 11000, 11000, 18000]

--- STARTING HAND 9 ---
Hand 9 Results: [-1000, 1000, 0]
New Stacks: [0, 10000, 12000, 18000]

--- STARTING HAND 10 ---


KeyboardInterrupt: 